In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

df = pd.read_csv('../data/GlobalWeatherRepository.csv')
print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/GlobalWeatherRepository.csv'

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

# Visualize
if len(missing) > 0:
    plt.figure(figsize=(10, 5))
    missing.plot(kind='bar')
    plt.title('Missing Values by Column')
    plt.ylabel('Count')
    plt.show()
else:
    print("No missing values found.")

In [ ]:
# Numeric columns: fill with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Categorical columns: fill with mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining missing values:", df.isnull().sum().sum())

In [ ]:
df['last_updated'] = pd.to_datetime(df['last_updated'])
df = df.sort_values('last_updated')
print(df['last_updated'].min(), "to", df['last_updated'].max())

In [ ]:
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower) | (data[column] > upper)]
    return outliers, lower, upper

key_cols = ['temperature_celsius', 'precip_mm', 'humidity', 'wind_kph', 'pressure_mb']
key_cols = [c for c in key_cols if c in df.columns]

for col in key_cols:
    outliers, low, up = detect_outliers_iqr(df, col)
    print(f"{col}: {len(outliers)} outliers (bounds: {low:.2f} to {up:.2f})")

In [ ]:
def cap_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    data[column] = data[column].clip(lower, upper)
    return data

for col in key_cols:
    df = cap_outliers(df, col)

print("Outliers capped.")

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
normalized_cols = [c + '_norm' for c in key_cols]
df[normalized_cols] = scaler.fit_transform(df[key_cols])

df[key_cols + normalized_cols].head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(key_cols[:4]):
    sns.histplot(df[col], kde=True, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 10))
corr = df[key_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Between Weather Features')
plt.show()

In [ ]:
daily_temp = df.groupby(df['last_updated'].dt.date)['temperature_celsius'].mean()

plt.figure(figsize=(14, 6))
daily_temp.plot()
plt.title('Average Daily Temperature Over Time')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.show()

In [ ]:
daily_precip = df.groupby(df['last_updated'].dt.date)['precip_mm'].mean()

plt.figure(figsize=(14, 6))
daily_precip.plot(color='steelblue')
plt.title('Average Daily Precipitation Over Time')
plt.xlabel('Date')
plt.ylabel('Precipitation (mm)')
plt.show()

In [ ]:
top_countries = df.groupby('country')['temperature_celsius'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
top_countries.plot(kind='barh', color='coral')
plt.title('Top 10 Hottest Countries (Average Temperature)')
plt.xlabel('Temperature (°C)')
plt.show()

In [ ]:
ts = df.groupby(df['last_updated'].dt.date)['temperature_celsius'].mean()
ts.index = pd.to_datetime(ts.index)
ts = ts.asfreq('D').interpolate()  # fill any gap days

print(ts.head())
print("Length:", len(ts))

In [ ]:
train_size = int(len(ts) * 0.8)
train, test = ts[:train_size], ts[train_size:]

plt.figure(figsize=(14, 6))
train.plot(label='Train')
test.plot(label='Test')
plt.legend()
plt.title('Train/Test Split')
plt.show()

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(train, order=(2, 1, 2))
model_fit = model.fit()
print(model_fit.summary())

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

forecast = model_fit.forecast(steps=len(test))

mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
mape = np.mean(np.abs((test.values - forecast.values) / test.values)) * 100

print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAPE: {mape:.2f}%")

plt.figure(figsize=(14, 6))
train.plot(label='Train')
test.plot(label='Actual')
forecast.plot(label='Forecast')
plt.legend()
plt.title('ARIMA Forecast vs Actual')
plt.show()

In [5]:
import pandas as pd
print(pd.__version__)

2.3.3
